In [38]:
import pandas as pd

TimeRange 系列函数的功能是找出每个电话号码在每个基站的进出时间对 (enter_time, exit_time)，有 robust 与 simple 版本的原因是我突然害怕基站出什么故障，导致某个人在某个基站会进去两次而只出来一次，或者进去一次，出来两次，于是有了 robust 版本，如果某人在某站出现 enter1, enter2, exit 序列，我们就保守取进出时间对 (enter1, exit)，如果出现 enter, exit1, exit2 序列，就保守取 (enter, exit2) 时间对。而 simple 模式就假定数据不会有异常。

In [ ]:
def TimeRangeRobust(dfinfo, Phone=True):
    df = dfinfo.sort_values(
        by=["phone", "cellid", "time(s)"]
    ).copy()
    all_intervals = []
    groups = df.groupby(["phone", "cellid"])
    for (phone, cellid), group in groups:
        # 这个循环表示一个用户在一个基站的所有记录，按照时间排序后，找到所有的进入和离开时间，并且合并连续的进入和离开时间段
        group = group.sort_values("time(s)")
        inside = False
        enter_time = None
        last_exit_time = None
        rows = group.to_dict("records")
        n = len(rows)
        for i, row in enumerate(rows):
            t = row["time(s)"]
            reg = row["register"]
            # ENTER
            if reg == 1:
                if not inside:
                    inside = True
                    enter_time = t
                    last_exit_time = None
            # EXIT
            elif reg == 2:
                if inside:
                    last_exit_time = t
                    next_is_enter = False
                    if i == n - 1:
                        next_is_enter = True
                    else:
                        next_reg = rows[i + 1]["register"]
                        if next_reg == 1:
                            next_is_enter = True
                    if next_is_enter:
                        # Phone == False 的情况别管了罢，后面是要用到 phone 的
                        if Phone == True:
                            all_intervals.append({
                                "phone": phone,
                                "cellid": cellid,
                                "enter_time": enter_time,
                                "exit_time": last_exit_time
                            })
                        else:
                            all_intervals.append({
                                "cellid": cellid,
                                "enter_time": enter_time,
                                "exit_time": last_exit_time
                            })
                        inside = False
                        enter_time = None
                        last_exit_time = None
    all_intervals_df = pd.DataFrame(all_intervals)
    return all_intervals_df

In [ ]:
def TimeRangeSimple(dfinfo, Phone=True):
    # 这个方法简单地将每个进入和离开记录配对，可能会有一些异常情况没有处理，比如连续的进入或者连续的离开，这些情况在实际数据中可能存在，但在这个简单版本中不做处理
    df = dfinfo.sort_values(
        by=["phone", "cellid", "time(s)"]
    ).copy()

    # 进入记录
    enter_df = df[df["register"] == 1].copy()
    enter_df = enter_df.rename(columns={"time(s)": "enter_time"})

    # 离开记录
    exit_df = df[df["register"] == 2].copy()
    exit_df = exit_df.rename(columns={"time(s)": "exit_time"})

    # 给每个手机号+基站内部的进出记录编号
    # 用于一一配对
    enter_df["idx"] = enter_df.groupby(
        ["phone", "cellid"]
    ).cumcount()

    exit_df["idx"] = exit_df.groupby(
        ["phone", "cellid"]
    ).cumcount()

    # 合并进入与离开
    result = pd.merge(
        enter_df[["phone", "cellid", "enter_time", "idx"]],
        exit_df[["phone", "cellid", "exit_time", "idx"]],
        on=["phone", "cellid", "idx"],
        how="inner"
    )

    # 最终结果
    if Phone == True:
        result = result[["phone", "cellid", "enter_time", "exit_time"]]
    else:
        result = result[["cellid", "enter_time", "exit_time"]]

    return result

In [ ]:
def TimeRange(dfinfo, Phone=True, Robust=True):
    if Robust:
        return TimeRangeRobust(dfinfo, Phone)
    else:
        return TimeRangeSimple(dfinfo, Phone)
    

In [ ]:
def FindContacts(interval_infected, interval_all, cdinfo_infected, include_infected=False):
    # 这个函数的作用是找到所有与感染者在同一基站且时间重叠的用户，并将他们的手机号导出到 contacts.txt 文件中
    contacts = set()
    for _, infected in interval_infected.iterrows():
        infected_cell = infected["cellid"]
        infected_start = infected["enter_time"]
        infected_end = infected["exit_time"]
        # 同基站
        same_cell = interval_all[
            interval_all["cellid"] == infected_cell
        ]
        # 区间重叠判定，(a, b) 和 (c, d) 重叠的条件是 a <= d and b >= c
        overlap = same_cell[
            (same_cell["enter_time"] <= infected_end)
            &
            (same_cell["exit_time"] >= infected_start)
        ]
        # 加入手机号
        contacts.update(overlap["phone"].tolist())
    if not include_infected:
        # 这里就看需求了，如果不包含感染者自己，那么就需要把感染者的手机号从 contacts 中去掉
        # 感染者手机号集合
        infected_phones = set(
            cdinfo_infected["phone"].unique()
        )
        # 去掉感染者自己
        contacts = list(contacts - infected_phones)
    # 升序排序
    contacts = sorted(contacts)
    # 导出 txt
    with open("contacts.txt", "w", encoding="utf-8") as f:
        for phone in contacts:
            f.write(f"{phone}\n")

    print("已导出 contacts.txt")

In [ ]:
cdinfo = pd.read_csv("cdinfo_fixed.txt", delimiter=",", header=None, names=["cellid","time(s)","register", "phone"])
infected = pd.read_csv("infected.txt", delimiter=",", header=None, names=["phone"])
cdinfo_infected = cdinfo[cdinfo["phone"].isin(infected["phone"])]
interval_infected = TimeRange(cdinfo_infected, Phone=True, Robust=True)
interval_all = TimeRange(cdinfo, Phone=True, Robust=True)
FindContacts(interval_infected, interval_all, cdinfo_infected, include_infected=False)